# Módulo 4 · Consultas SQL con Amazon Athena (boto3)


| # | Ejercicio | En la consola |
|---|---|---|
| 1 | Crear un workgroup con carpeta de resultados y tope de escaneo | Athena → Workgroups / Settings → Manage |
| 2 | Primeras consultas: ¿se ven los datos y las 8 particiones? | Query editor |
| 3 | Consultas de análisis: ventas por canal y top de productos | Query editor |
| 4 | Experimento: cuánto se escanea con y sin filtros | Panel *Data scanned* |
| 5 | Tu turno: categoría × canal y mejor cliente | Query editor |
| 6 | Funciones de ventana: ranking por categoría | Query editor |
| 7 | CTAS: guardar un resultado como Parquet en `curated/` | Query editor |
| 8 | Vistas: guardar la consulta, no los datos | Query editor |
| 9 | Resumen de lo escaneado y su costo | — |
| 10 | Limpieza (opcional) | — |

**Requisito:** haber ejecutado `02_glue_crawler_catalogo.ipynb`. La tabla `ventas_db.ventas` debe existir en el Data Catalog.

**Costo:** Athena cobra **USD 5 por TB escaneado, con mínimo de 10 MB por consulta**. Con nuestro dataset de pocos KB cada consulta cuesta unos USD 0.00005. Las sentencias DDL (`CREATE VIEW`, `DROP TABLE`, `SHOW ...`) no se cobran.

## Repaso: ¿cómo funciona Athena?

```
 Tu SQL ──► Athena (motor Trino, sin servidores)
               │  1. pregunta al Glue Data Catalog: columnas, formato, ubicación y particiones de ventas
               │  2. lee de S3 solo las particiones y columnas que necesita
               ▼
            Resultado CSV en s3://<bucket>/athena-results/
```

- No hay cluster que encender ni apagar: pagas solo por los **bytes escaneados**.
- Athena **siempre** escribe el resultado en S3. Sin una ubicación de resultados no deja ejecutar consultas.

## Configuración

Mismo `BUCKET` y `REGION` de los notebooks anteriores.

In [2]:
from pathlib import Path
import csv
import time

import boto3
import pandas as pd
from botocore.exceptions import ClientError, ProfileNotFound

BUCKET = "de-tecsup-marcell"            # <- el mismo del notebook 01
REGION = "us-east-2"
PERFIL = "curso"

DATABASE = "ventas_db"
WORKGROUP = "curso-de"
RESULTADOS = f"s3://{BUCKET}/athena-results/"
TOPE_MB = 100                           # tope de datos escaneados por consulta

PROYECTO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
CREDENCIALES = PROYECTO / "credentials"

In [3]:
def leer_csv_credenciales(carpeta: Path) -> dict:
    """Lee el .csv que descarga IAM (columnas 'Access key ID' y 'Secret access key')."""
    archivo = next(carpeta.glob("*accessKeys*.csv"))
    with archivo.open(encoding="utf-8-sig") as f:
        fila = next(csv.DictReader(f))
    return {
        "aws_access_key_id": fila["Access key ID"],
        "aws_secret_access_key": fila["Secret access key"],
    }


try:
    session = boto3.Session(profile_name=PERFIL, region_name=REGION)
    origen = f"perfil '{PERFIL}' de ~/.aws/"
except ProfileNotFound:
    session = boto3.Session(**leer_csv_credenciales(CREDENCIALES), region_name=REGION)
    origen = "archivo .csv de credentials/"

print("Credenciales tomadas de:", origen)
print("ARN:", session.client("sts").get_caller_identity()["Arn"])

s3 = session.client("s3")
glue = session.client("glue")
athena = session.client("athena")

tablas = [t["Name"] for t in glue.get_tables(DatabaseName=DATABASE)["TableList"]]
print(f"Tablas en {DATABASE}:", tablas)
assert "ventas" in tablas, "No existe ventas_db.ventas: ejecuta primero el notebook 02"

Credenciales tomadas de: archivo .csv de credentials/
ARN: arn:aws:iam::073813807844:user/marcell-de
Tablas en ventas_db: ['ventas', 'ventas_diarias']


## Ejercicio 1 · Workgroup con resultados y tope de escaneo

En la consola harías: *Settings → Manage* para elegir la carpeta de resultados y, opcionalmente, crear un workgroup con límite.

Un **workgroup** agrupa consultas y les aplica una configuración común:

| Opción | Valor | Para qué |
|---|---|---|
| `ResultConfiguration.OutputLocation` | `s3://<bucket>/athena-results/` | Dónde Athena guarda el CSV de cada resultado |
| `BytesScannedCutoffPerQuery` | 100 MB | **Red de seguridad:** Athena cancela cualquier consulta que intente escanear más. El mínimo permitido es 10 MB |
| `EnforceWorkGroupConfiguration` | `False` | Si fuera `True`, ningún cliente podría cambiar la ubicación de resultados, y Athena **rechazaría los CTAS con `external_location`** (ejercicio 7). El tope de escaneo se aplica igual |
| `PublishCloudWatchMetricsEnabled` | `True` | Métricas de uso en CloudWatch |

In [4]:
config_wg = {
    "ResultConfiguration": {"OutputLocation": RESULTADOS},
    "EnforceWorkGroupConfiguration": False,   # True impide usar external_location en CTAS (ejercicio 7)
    "PublishCloudWatchMetricsEnabled": True,
    "BytesScannedCutoffPerQuery": TOPE_MB * 1024 * 1024,
    "EngineVersion": {"SelectedEngineVersion": "AUTO"},
}

existentes = [w["Name"] for w in athena.list_work_groups()["WorkGroups"]]
if WORKGROUP in existentes:
    athena.update_work_group(
        WorkGroup=WORKGROUP,
        ConfigurationUpdates={
            "ResultConfigurationUpdates": {"OutputLocation": RESULTADOS},
            "EnforceWorkGroupConfiguration": False,   # True impide usar external_location en CTAS (ejercicio 7)
            "PublishCloudWatchMetricsEnabled": True,
            "BytesScannedCutoffPerQuery": TOPE_MB * 1024 * 1024,
        },
    )
    print("Workgroup actualizado:", WORKGROUP)
else:
    athena.create_work_group(
        Name=WORKGROUP,
        Configuration=config_wg,
        Description="Workgroup del curso de Data Engineering con tope de escaneo",
    )
    print("Workgroup creado:", WORKGROUP)

wg = athena.get_work_group(WorkGroup=WORKGROUP)["WorkGroup"]["Configuration"]
print("Resultados en:", wg["ResultConfiguration"]["OutputLocation"])
print("Tope por consulta:", wg["BytesScannedCutoffPerQuery"] / 1024 / 1024, "MB")
print("Motor:", wg["EngineVersion"]["EffectiveEngineVersion"])
print("Forzar configuración:", wg["EnforceWorkGroupConfiguration"])

Workgroup creado: curso-de
Resultados en: s3://de-tecsup-marcell/athena-results/
Tope por consulta: 100.0 MB
Motor: Athena engine version 3
Forzar configuración: False


### Una función para ejecutar SQL desde Python

Con boto3, una consulta de Athena es **asíncrona**: se envía con `start_query_execution`, se espera hasta que termine y luego se leen los resultados. La función `consultar()` hace los tres pasos, devuelve un DataFrame e imprime lo mismo que el panel de la consola: tiempo y datos escaneados.

Además va guardando un **historial** para el resumen de costos del ejercicio 9.

In [5]:
HISTORIAL = []
NUMERICOS = {"tinyint", "smallint", "integer", "bigint", "double", "float", "real", "decimal"}


def tamano_legible(n: float) -> str:
    for unidad in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024:
            return f"{n:.1f} {unidad}"
        n /= 1024
    return f"{n:.1f} PB"


def consultar(sql: str, etiqueta: str = "") -> pd.DataFrame:
    """Ejecuta SQL en Athena, espera a que termine y devuelve el resultado como DataFrame."""
    qid = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": DATABASE},
        WorkGroup=WORKGROUP,
    )["QueryExecutionId"]

    # 1. Esperar: QUEUED -> RUNNING -> SUCCEEDED | FAILED | CANCELLED
    while True:
        ejecucion = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]
        estado = ejecucion["Status"]["State"]
        if estado in ("SUCCEEDED", "FAILED", "CANCELLED"):
            break
        time.sleep(1)

    if estado != "SUCCEEDED":
        raise RuntimeError(f"{estado}: {ejecucion['Status'].get('StateChangeReason')}")

    # 2. Estadísticas: lo mismo que muestra la consola
    stats = ejecucion["Statistics"]
    escaneado = stats.get("DataScannedInBytes", 0)
    tipo = ejecucion["StatementType"]                 # DML (SELECT, CTAS) | DDL | UTILITY
    HISTORIAL.append({
        "consulta": etiqueta or sql.strip().splitlines()[0][:60],
        "tipo": tipo,
        "bytes": escaneado,
        "segundos": stats["TotalExecutionTimeInMillis"] / 1000,
    })
    print(f"✔ {tipo} · {stats['TotalExecutionTimeInMillis'] / 1000:.1f} s · "
          f"datos escaneados: {tamano_legible(escaneado)}")

    # 3. Leer los resultados (paginados de 1000 en 1000 filas)
    columnas, tipos, filas = None, None, []
    for pagina in athena.get_paginator("get_query_results").paginate(QueryExecutionId=qid):
        rs = pagina["ResultSet"]
        if columnas is None:
            info = rs["ResultSetMetadata"]["ColumnInfo"]
            columnas = [c["Name"] for c in info]
            tipos = [c["Type"] for c in info]
            rows = rs["Rows"]
            # En un SELECT la 1.ª fila repite los nombres de las columnas: la saltamos
            if rows and [d.get("VarCharValue") for d in rows[0]["Data"]] == columnas:
                rows = rows[1:]
        else:
            rows = rs["Rows"]
        filas.extend([d.get("VarCharValue") for d in r["Data"]] for r in rows)

    df = pd.DataFrame(filas, columns=columnas)
    for col, t in zip(columnas or [], tipos or []):
        if t in NUMERICOS:                            # Athena devuelve todo como texto
            df[col] = pd.to_numeric(df[col])
    return df

## Ejercicio 2 · Primeras consultas

Las dos consultas de la diapositiva 28: ¿se ven los datos? ¿Están las 8 particiones?

In [6]:
consultar("SELECT * FROM ventas LIMIT 10", "SELECT * LIMIT 10")

✔ DML · 1.6 s · datos escaneados: 16.5 KB


,producto_id,nombre,categoria,precio_unitario,cantidad,pedido_id,fecha,cliente_id,canal,monto,year,month,day
0,1,Laptop,Tecnologia,3200.0,4,ERP-20261023-001,2026-10-23,85,tienda,12800.0,2026,10,23
1,6,Escritorio,Muebles,540.0,4,ERP-20261021-001,2026-10-21,82,tienda,2160.0,2026,10,21
2,9,Mochila,Accesorios,150.0,4,ERP-20261025-003,2026-10-25,89,tienda,600.0,2026,10,25
3,6,Escritorio,Muebles,540.0,2,ERP-20261024-001,2026-10-24,84,tienda,1080.0,2026,10,24
4,3,Teclado,Tecnologia,120.0,4,ERP-20261020-001,2026-10-20,92,tienda,480.0,2026,10,20
5,2,Mouse,Tecnologia,45.0,4,ERP-20261027-001,2026-10-27,57,tienda,180.0,2026,10,27
6,8,Lapicero,Libreria,2.5,2,ERP-20261026-001,2026-10-26,44,tienda,5.0,2026,10,26
7,3,Teclado,Tecnologia,120.0,4,ERP-20261022-001,2026-10-22,2,tienda,480.0,2026,10,22
8,3,Teclado,Tecnologia,120.0,1,ERP-20261024-001,2026-10-24,84,tienda,120.0,2026,10,24
9,6,Escritorio,Muebles,540.0,2,ERP-20261024-002,2026-10-24,58,tienda,1080.0,2026,10,24


In [7]:
consultar("""
SELECT year, month, day, COUNT(*) AS filas
FROM ventas
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3
""", "filas por partición")

✔ DML · 0.9 s · datos escaneados: 0.0 B


,year,month,day,filas
0,2026,10,20,75
1,2026,10,21,73
2,2026,10,22,100
3,2026,10,23,107
4,2026,10,24,78
5,2026,10,25,85
6,2026,10,26,69
7,2026,10,27,91


También se pueden listar las particiones registradas en el catálogo. Es una sentencia de utilidad: no escanea datos.

In [8]:
consultar("SHOW PARTITIONS ventas", "SHOW PARTITIONS")

✔ UTILITY · 0.8 s · datos escaneados: 0.0 B


,partition
0,year=2026/month=10/day=27
1,year=2026/month=10/day=23
2,year=2026/month=10/day=24
3,year=2026/month=10/day=20
4,year=2026/month=10/day=22
5,year=2026/month=10/day=21
6,year=2026/month=10/day=25
7,year=2026/month=10/day=26


## Ejercicio 3 · Consultas de análisis

Recuerda: las columnas de partición son `string`, así que se filtran **con comillas**: `year = '2026'`.

In [11]:
consultar("""
-- 1. Ventas por canal en un día (lee 1 partición)
SELECT canal,
       COUNT(DISTINCT pedido_id) AS pedidos,
       ROUND(SUM(monto), 2)      AS monto_total
FROM ventas_db.ventas
WHERE year = '2026' AND month = '10' AND day = '22'
GROUP BY canal
ORDER BY monto_total DESC
""", "ventas por canal (1 día)")

✔ DML · 0.6 s · datos escaneados: 878.0 B


,canal,pedidos,monto_total
0,tienda,25,88295.0
1,web,27,59777.5


In [12]:
consultar("""
-- 2. Top 5 productos del mes por unidades
SELECT nombre, SUM(cantidad) AS unidades
FROM ventas_db.ventas
WHERE year = '2026' AND month = '10'
GROUP BY nombre
ORDER BY unidades DESC
LIMIT 5
""", "top 5 productos")

✔ DML · 0.7 s · datos escaneados: 2.9 KB


,nombre,unidades
0,Silla ergonomica,193
1,Cuaderno,187
2,Audifonos,178
3,Mochila,175
4,Lapicero,175


## Ejercicio 4 · Experimento: ¿cuánto se escanea?

En la consola se compara el panel *Data scanned*. Aquí lo medimos con `DataScannedInBytes` sobre la misma pregunta, escrita de cuatro formas:

| Variante | Particiones que lee | Columnas que lee |
|---|---|---|
| A | 1 día | solo `canal` y `monto` |
| B | los 8 días (sin filtro de `day`) | solo `canal` y `monto` |
| C | 1 día | **todas** (`SELECT *`) |
| D | los 8 días | **todas** |

In [13]:
variantes = {
    "A · 1 día, 2 columnas":  "SELECT canal, SUM(monto) AS monto FROM ventas WHERE year='2026' AND month='10' AND day='22' GROUP BY canal",
    "B · 8 días, 2 columnas": "SELECT canal, SUM(monto) AS monto FROM ventas GROUP BY canal",
    "C · 1 día, SELECT *":    "SELECT * FROM ventas WHERE year='2026' AND month='10' AND day='22'",
    "D · 8 días, SELECT *":   "SELECT * FROM ventas",
}

experimento = []
for nombre, sql in variantes.items():
    consultar(sql, nombre)
    experimento.append({"variante": nombre, "bytes escaneados": HISTORIAL[-1]["bytes"]})

experimento = pd.DataFrame(experimento)
experimento["vs. A"] = (experimento["bytes escaneados"] / experimento["bytes escaneados"].iloc[0]).round(1).astype(str) + "x"
experimento

✔ DML · 0.5 s · datos escaneados: 443.0 B
✔ DML · 0.7 s · datos escaneados: 3.3 KB
✔ DML · 1.3 s · datos escaneados: 2.2 KB
✔ DML · 1.5 s · datos escaneados: 16.5 KB


,variante,bytes escaneados,vs. A
0,"A · 1 día, 2 columnas",443,1.0x
1,"B · 8 días, 2 columnas",3373,7.6x
2,"C · 1 día, SELECT *",2268,5.1x
3,"D · 8 días, SELECT *",16876,38.1x


**Qué observar:**

- **A frente a B (partition pruning):** sin el filtro de `day`, Athena lee las 8 carpetas en vez de 1, unas 8 veces más datos.
- **A frente a C (formato columnar):** Parquet guarda cada columna por separado, así que `SELECT *` lee las 10 columnas en vez de 2.
- **D** combina ambos errores: es el peor caso.

Con nuestros pocos KB, las 4 variantes se cobran igual, porque el mínimo es **10 MB por consulta**. Con 1 TB de ventas la diferencia entre A y D serían cientos de dólares por consulta.

> `LIMIT` **no** reduce lo escaneado de forma fiable: Athena puede leer archivos completos antes de cortar. Lo que ahorra es filtrar particiones y elegir columnas.

## Ejercicio 5 · Tu turno

1. Monto total por **categoría y canal** de toda la semana.
2. El **cliente con mayor compra** (monto total) de la semana.

Intenta escribirlas antes de ejecutar las soluciones de abajo.

In [14]:
# Solución 1: monto por categoría y canal
consultar("""
SELECT categoria,
       canal,
       COUNT(DISTINCT pedido_id) AS pedidos,
       ROUND(SUM(monto), 2)      AS monto_total
FROM ventas_db.ventas
WHERE year = '2026' AND month = '10' AND day BETWEEN '20' AND '27'
GROUP BY categoria, canal
ORDER BY categoria, monto_total DESC
""", "categoría x canal")

✔ DML · 0.9 s · datos escaneados: 7.4 KB


,categoria,canal,pedidos,monto_total
0,Accesorios,tienda,65,34040.0
1,Accesorios,web,63,31370.0
2,Libreria,tienda,69,1547.5
3,Libreria,web,59,1227.5
4,Muebles,web,61,107160.0
5,Muebles,tienda,65,104310.0
6,Tecnologia,tienda,115,358905.0
7,Tecnologia,web,97,341755.0


In [17]:
# Solución 2: cliente con mayor compra
consultar("""
SELECT cliente_id,
       COUNT(DISTINCT pedido_id) AS pedidos,
       ROUND(SUM(monto), 2)      AS monto_total
FROM ventas_db.ventas
WHERE year = '2026' AND month = '10'
GROUP BY cliente_id
ORDER BY monto_total DESC
LIMIT 10
""", "mejor cliente")

✔ DML · 0.6 s · datos escaneados: 8.1 KB


,cliente_id,pedidos,monto_total
0,42,8,43495.0
1,90,11,35787.5
2,17,5,28240.0
3,49,4,27500.0
4,73,3,25842.5
5,85,6,22790.0
6,11,2,22650.0
7,87,6,21467.5
8,28,4,21240.0
9,69,4,20995.0


## Ejercicio 6 · Funciones de ventana (diapositiva 30)

`RANK() OVER (PARTITION BY categoria ORDER BY ...)` numera los productos **dentro de cada categoría**. A diferencia de `GROUP BY`, no colapsa las filas.

> Ojo: el `PARTITION BY` de una ventana no tiene nada que ver con las particiones de S3. Solo define los grupos sobre los que se calcula el ranking.

In [18]:
consultar("""
SELECT categoria,
       nombre,
       ROUND(SUM(monto), 2) AS monto,
       RANK() OVER (
         PARTITION BY categoria
         ORDER BY SUM(monto) DESC
       ) AS puesto
FROM ventas_db.ventas
WHERE year = '2026' AND month = '10'
GROUP BY categoria, nombre
ORDER BY categoria, puesto
""", "ranking por categoría")

✔ DML · 0.6 s · datos escaneados: 5.6 KB


,categoria,nombre,monto,puesto
0,Accesorios,Audifonos,39160.0,1
1,Accesorios,Mochila,26250.0,2
2,Libreria,Cuaderno,2337.5,1
3,Libreria,Lapicero,437.5,2
4,Muebles,Silla ergonomica,133170.0,1
5,Muebles,Escritorio,78300.0,2
6,Tecnologia,Laptop,547200.0,1
7,Tecnologia,Monitor,128350.0,2
8,Tecnologia,Teclado,18720.0,3
9,Tecnologia,Mouse,6390.0,4


## Ejercicio 7 · CTAS: guardar el resultado como Parquet

`CREATE TABLE ... AS SELECT` hace dos cosas a la vez:

1. **Escribe** el resultado en S3, en el formato que elijas (aquí Parquet, en `curated/resumen_cliente/`).
2. **Registra** la tabla en el Data Catalog, lista para consultarse.

Es una mini-ETL hecha solo con SQL: lee de `processed/` y escribe en `curated/`.

**Para poder repetir el ejercicio:** CTAS falla si la tabla ya existe o si la carpeta de destino no está vacía. Por eso primero borramos la tabla (`DROP TABLE` solo borra metadatos) y luego los archivos de S3.

> **¿Por qué el workgroup no fuerza su configuración?** Con `EnforceWorkGroupConfiguration = True`, Athena guarda **todos** los resultados en la carpeta del workgroup, también los de un CTAS (en `athena-results/tables/<id>/`), y rechaza `external_location`. Como queremos que la tabla quede en `curated/`, en el ejercicio 1 lo dejamos en `False`. En una empresa se suele mantener en `True` y se definen ubicaciones por workgroup o por equipo.

In [19]:
DESTINO_PREFIJO = "curated/resumen_cliente/"

consultar("DROP TABLE IF EXISTS ventas_db.resumen_cliente", "DROP resumen_cliente")

viejos = s3.list_objects_v2(Bucket=BUCKET, Prefix=DESTINO_PREFIJO).get("Contents", [])
if viejos:
    s3.delete_objects(Bucket=BUCKET, Delete={"Objects": [{"Key": o["Key"]} for o in viejos]})
    print(f"Borrados {len(viejos)} archivo(s) previos de {DESTINO_PREFIJO}")

✔ DDL · 0.4 s · datos escaneados: 0.0 B


In [20]:
consultar(f"""
CREATE TABLE ventas_db.resumen_cliente
WITH (format = 'PARQUET',
      write_compression = 'SNAPPY',
      external_location = 's3://{BUCKET}/{DESTINO_PREFIJO}')
AS
SELECT cliente_id,
       COUNT(DISTINCT pedido_id) AS pedidos,
       SUM(monto)                AS monto_total
FROM ventas_db.ventas
GROUP BY cliente_id
""", "CTAS resumen_cliente")

✔ DDL · 1.0 s · datos escaneados: 8.1 KB


,rows


¿Qué quedó en S3? Archivos Parquet que escribió Athena, con nombres generados:

In [21]:
for o in s3.list_objects_v2(Bucket=BUCKET, Prefix=DESTINO_PREFIJO).get("Contents", []):
    print(f"{tamano_legible(o['Size']):>10}  {o['Key']}")

    1.8 KB  curated/resumen_cliente/20261114_161424_00106_7vu3n_9aeeaca8-dfa5-435b-a5fd-8a8e1d1d849b


Y la nueva tabla ya se consulta como cualquier otra:

In [22]:
consultar("""
SELECT *
FROM ventas_db.resumen_cliente
ORDER BY monto_total DESC
LIMIT 10
""", "top 10 resumen_cliente")

✔ DML · 0.6 s · datos escaneados: 1.3 KB


,cliente_id,pedidos,monto_total
0,42,8,43495.0
1,90,11,35787.5
2,17,5,28240.0
3,49,4,27500.0
4,73,3,25842.5
5,85,6,22790.0
6,11,2,22650.0
7,87,6,21467.5
8,28,4,21240.0
9,69,4,20995.0


## Ejercicio 8 · Vistas: guardan la consulta, no los datos

| | CTAS | Vista |
|---|---|---|
| ¿Guarda datos en S3? | Sí, Parquet en `curated/` | No |
| ¿Se recalcula? | No: es una foto del momento en que se creó | Sí, cada vez que se consulta |
| ¿Cuánto escanea al consultarla? | Solo la tabla resultado (pequeña) | La tabla original (`ventas`) |
| Úsala para | Resultados pesados que se leen muchas veces | Simplificar consultas y dar acceso a un subconjunto |

In [23]:
consultar("""
CREATE OR REPLACE VIEW ventas_db.v_web AS
SELECT * FROM ventas_db.ventas
WHERE canal = 'web'
""", "CREATE VIEW v_web")

✔ DDL · 0.4 s · datos escaneados: 0.0 B


""


In [25]:
consultar("""
SELECT day, COUNT(*) AS filas, ROUND(SUM(monto), 2) AS monto_web
FROM ventas_db.v_web
GROUP BY day
ORDER BY day
""", "consulta sobre v_web")

✔ DML · 0.9 s · datos escaneados: 3.6 KB


,day,filas,monto_web
0,20,46,59695.0
1,21,27,54882.5
2,22,48,59777.5
3,23,45,57415.0
4,24,29,62777.5
5,25,47,96662.5
6,26,40,53080.0
7,27,40,37222.5


En el Data Catalog la vista aparece como una tabla de tipo `VIRTUAL_VIEW`: sin ubicación en S3, solo la consulta guardada.

In [26]:
for t in glue.get_tables(DatabaseName=DATABASE)["TableList"]:
    print(f"{t['Name']:<16} {t.get('TableType'):<16} {t['StorageDescriptor'].get('Location', '') or '(sin datos en S3)'}")

resumen_cliente  EXTERNAL_TABLE   s3://de-tecsup-marcell/curated/resumen_cliente/
v_web            VIRTUAL_VIEW     (sin datos en S3)
ventas           EXTERNAL_TABLE   s3://de-tecsup-marcell/processed/ventas/
ventas_diarias   EXTERNAL_TABLE   s3://de-tecsup-marcell/curated/ventas_diarias


## Ejercicio 9 · ¿Cuánto escaneamos y cuánto costó?

Athena cobra **USD 5 por TB**, redondeando cada consulta DML a un mínimo de **10 MB**. Las DDL y de utilidad no se cobran.

In [27]:
historial = pd.DataFrame(HISTORIAL)
cobrable = historial["tipo"] == "DML"
historial["bytes cobrados"] = historial["bytes"].clip(lower=10 * 1024**2).where(cobrable, 0)
historial["costo USD"] = historial["bytes cobrados"] / 1024**4 * 5

print(f"Consultas ejecutadas:  {len(historial)} ({cobrable.sum()} cobrables)")
print(f"Datos escaneados:      {tamano_legible(historial['bytes'].sum())}")
print(f"Costo estimado total:  USD {historial['costo USD'].sum():.5f}")
historial

Consultas ejecutadas:  22 (18 cobrables)
Datos escaneados:      103.6 KB
Costo estimado total:  USD 0.00086


,consulta,tipo,bytes,segundos,bytes cobrados,costo USD
0,SELECT * LIMIT 10,DML,16876,1.647,10485760,0.000048
1,filas por partición,DML,0,0.912,10485760,0.000048
2,SHOW PARTITIONS,UTILITY,0,0.840,0,0.000000
3,ventas por canal (1 día),DML,878,0.711,10485760,0.000048
4,ventas por canal (1 día),DML,6419,0.704,10485760,0.000048
5,ventas por canal (1 día),DML,878,0.648,10485760,0.000048
6,top 5 productos,DML,2998,0.693,10485760,0.000048
7,"A · 1 día, 2 columnas",DML,443,0.531,10485760,0.000048
8,"B · 8 días, 2 columnas",DML,3373,0.668,10485760,0.000048
9,"C · 1 día, SELECT *",DML,2268,1.277,10485760,0.000048


### Buenas prácticas de costo y rendimiento

| ✅ Haz | ❌ Evita |
|---|---|
| Filtra siempre por columnas de partición | `SELECT *` sobre tablas grandes |
| Selecciona solo las columnas que necesitas | CSV sin comprimir para análisis frecuente |
| Usa Parquet u ORC comprimido | Miles de archivos de pocos KB |
| Apunta a archivos de 128 MB o más | Confiar en `LIMIT` para ahorrar escaneo |
| Fija un tope por workgroup | Particionar por columnas de alta cardinalidad (ej. `cliente_id`) |

## Ejercicio 10 · Limpieza (opcional)

Borra la vista, la tabla CTAS y sus archivos, los resultados de Athena y el workgroup. **No** toca `ventas`, que pertenece al notebook 02.

> **Orden de limpieza:** de atrás hacia adelante, **04 → 03 → 02 → 01**. El bucket (notebook 01) se borra al final, porque Athena necesita su carpeta `athena-results/` incluso para ejecutar un `DROP`.

In [ ]:
BORRAR = True
DESTINO_PREFIJO = "curated/resumen_cliente/"


def borrar_prefijo(prefijo: str) -> None:
    try:
        for pagina in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=prefijo):
            objetos = [{"Key": o["Key"]} for o in pagina.get("Contents", [])]
            if objetos:
                s3.delete_objects(Bucket=BUCKET, Delete={"Objects": objetos})
        print("Archivos borrados en", prefijo)
    except ClientError as e:
        if e.response["Error"]["Code"] != "NoSuchBucket":
            raise
        print(f"El bucket {BUCKET} ya no existe: nada que borrar en {prefijo}")


if BORRAR:
    # Borramos con la API de Glue (no con DROP en Athena): así no hace falta el bucket de resultados
    for nombre in ["v_web", "resumen_cliente"]:
        try:
            glue.delete_table(DatabaseName=DATABASE, Name=nombre)
            print("Borrada del catálogo:", nombre)
        except glue.exceptions.EntityNotFoundException:
            print("Ya no existía en el catálogo:", nombre)

    borrar_prefijo(DESTINO_PREFIJO)           # borrar la tabla no borra los Parquet de S3
    borrar_prefijo("athena-results/")

    if WORKGROUP in [w["Name"] for w in athena.list_work_groups()["WorkGroups"]]:
        athena.delete_work_group(WorkGroup=WORKGROUP, RecursiveDeleteOption=True)
        print("Workgroup eliminado:", WORKGROUP)
    else:
        print("El workgroup ya no existía:", WORKGROUP)